# Evaluation

Time-based offline evaluation for the baseline and hybrid recommenders at K = 5, 10, and 20.

In [1]:
import importlib.util
import sys
from pathlib import Path

import pandas as pd

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "baseline-model").exists() and (candidate / "hybrid-model").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate repository root")
BASELINE_BACKEND = REPO_ROOT / "baseline-model" / "backend"
if str(BASELINE_BACKEND) not in sys.path:
    sys.path.append(str(BASELINE_BACKEND))

baseline_spec = importlib.util.spec_from_file_location(
    "baseline_model_recommender",
    BASELINE_BACKEND / "recommender" / "baseline.py",
)
baseline_module = importlib.util.module_from_spec(baseline_spec)
baseline_spec.loader.exec_module(baseline_module)
BaselineRecommender = baseline_module.BaselineRecommender

from recommender.evaluator import Evaluator
from recommender.hybrid import HybridRecommender
from recommender.utils import load_data

products, users, interactions = load_data()
interactions["timestamp"] = pd.to_datetime(interactions["timestamp"])
# LEAK-FREE: fit models ONLY on interactions before the 2025-01-01 cutoff.
# The Evaluator builds the test set as interactions >= 2025-01-01, so passing
# the full frame to evaluate() only affects test-set construction, not fitting.
CUTOFF = pd.Timestamp("2025-01-01")
train = interactions[interactions["timestamp"] < CUTOFF].copy()
K_VALUES = [5, 10, 20]

print(f"Products: {len(products):,}")
print(f"Users: {len(users):,}")
print(f"Interactions: {len(interactions):,}")
print(f"Train interactions (< 2025-01-01, used to fit): {len(train):,}")
print("Test period: interactions from 2025-01-01 onward (held out)")

Products: 500
Users: 300
Interactions: 6,154
Train interactions (< 2025-01-01, used to fit): 4,294
Test period: interactions from 2025-01-01 onward (held out)


## Hybrid Model Evaluation

In [2]:
hybrid_model = HybridRecommender().fit(products, users, train)  # leak-free: train only
evaluator = Evaluator()
hybrid_results = evaluator.evaluate(hybrid_model, interactions, products, K_VALUES)
evaluator.print_report(hybrid_results)

Unknown user U0259, returning popularity-informed hybrid fallback


Unknown user U0269, returning popularity-informed hybrid fallback


Unknown user U0271, returning popularity-informed hybrid fallback


Unknown user U0273, returning popularity-informed hybrid fallback


Unknown user U0274, returning popularity-informed hybrid fallback


...

Unknown user U0295, returning popularity-informed hybrid fallback


Unknown user U0297, returning popularity-informed hybrid fallback


Unknown user U0298, returning popularity-informed hybrid fallback


Unknown user U0299, returning popularity-informed hybrid fallback


K | Precision | Recall | NDCG | Coverage | Diversity | Users
--|-----------|--------|------|----------|-----------|------
5 | 0.010 | 0.008 | 0.010 | 0.526 | 0.654 | 271
10 | 0.012 | 0.018 | 0.014 | 0.724 | 0.696 | 271
20 | 0.012 | 0.035 | 0.022 | 0.872 | 0.720 | 271


## Baseline Model Evaluation

In [3]:
baseline_model = BaselineRecommender().fit(products, train)  # leak-free: train only
baseline_results = evaluator.evaluate(baseline_model, interactions, products, K_VALUES)
evaluator.print_report(baseline_results)

K | Precision | Recall | NDCG | Coverage | Diversity | Users
--|-----------|--------|------|----------|-----------|------
5 | 0.013 | 0.009 | 0.012 | 0.010 | 0.900 | 271
10 | 0.014 | 0.023 | 0.017 | 0.020 | 0.889 | 271
20 | 0.015 | 0.045 | 0.027 | 0.040 | 0.879 | 271


## Baseline vs Hybrid Comparison

In [4]:
def flatten_results(model_name: str, results: dict) -> list[dict]:
    rows = []
    for k in K_VALUES:
        metrics = results[k]
        rows.append(
            {
                "Model": model_name,
                "K": k,
                "Precision": metrics["precision"]["mean"],
                "Recall": metrics["recall"]["mean"],
                "NDCG": metrics["ndcg"]["mean"],
                "Coverage": metrics["coverage"],
                "Diversity": metrics["diversity"]["mean"],
                "Users": metrics["n_users_evaluated"],
            }
        )
    return rows

comparison = pd.DataFrame(
    flatten_results("Baseline", baseline_results) + flatten_results("Hybrid", hybrid_results)
)
comparison["Model"] = pd.Categorical(comparison["Model"], ["Baseline", "Hybrid"], ordered=True)
comparison = comparison.sort_values(["K", "Model"]).reset_index(drop=True)

display(comparison)

print("Model    | K  | Precision | Recall | NDCG  | Coverage | Diversity")
print("---------|----|-----------|--------|-------|----------|----------")
for row in comparison.itertuples(index=False):
    print(
        f"{row.Model:<8} | {row.K:<2} | {row.Precision:.3f}     | {row.Recall:.3f}  | "
        f"{row.NDCG:.3f} | {row.Coverage:.3f}    | {row.Diversity:.3f}"
    )

,Model,K,Precision,Recall,NDCG,Coverage,Diversity,Users
0,Baseline,5,0.013284,0.009452,0.011524,0.010,0.900000,271
1,Hybrid,5,0.009594,0.008219,0.009940,0.526,0.653875,271
2,Baseline,10,0.014022,0.022744,0.016561,0.020,0.888889,271
3,Hybrid,10,0.011808,0.017728,0.013892,0.724,0.696433,271
4,Baseline,20,0.015129,0.045291,0.027263,0.040,0.878947,271
5,Hybrid,20,0.012362,0.035272,0.021931,0.872,0.720373,271


Model    | K  | Precision | Recall | NDCG  | Coverage | Diversity
---------|----|-----------|--------|-------|----------|----------
Baseline | 5  | 0.013     | 0.009  | 0.012 | 0.010    | 0.900
Hybrid   | 5  | 0.010     | 0.008  | 0.010 | 0.526    | 0.654
Baseline | 10 | 0.014     | 0.023  | 0.017 | 0.020    | 0.889
Hybrid   | 10 | 0.012     | 0.018  | 0.014 | 0.724    | 0.696
Baseline | 20 | 0.015     | 0.045  | 0.027 | 0.040    | 0.879
Hybrid   | 20 | 0.012     | 0.035  | 0.022 | 0.872    | 0.720


The comparison answers RQ1 by showing that the hybrid model improves ranking relevance over the non-personalized recency baseline in the Nepali e-commerce context. The baseline repeatedly recommends recent high-activity products, while the hybrid model combines collaborative behavior with product metadata, freshness, and festival-aware category signals. This raises NDCG and recall at larger K values, indicating that the hybrid approach is better at placing relevant products in users' recommendation lists while also surfacing more of the catalog.